In [2]:
import sys
import os 

import numpy as np 
import pandas as pd 

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB

sys.path.append(os.path.abspath("../.."))
from src.results_manager import ResultsManager

rm = ResultsManager("naive_bayes")

In [2]:
from src.load_processed import load_processed_data

dataset = load_processed_data("../../data/processed/preprocessed.npz")

X_train = dataset["X_train"]
X_test = dataset["X_test"]

y_train = dataset["y_train"]
y_test = dataset["y_test"]

font_test = dataset["font_test"]
italic_test = dataset["italic_test"]
strength_test = dataset["strength_test"]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("Number of classes:", len(np.unique(y_train)))
print("Data type:", X_train.dtype)

Train shape: (666136, 400)
Test shape: (166534, 400)
Number of classes: 256
Data type: float32


In [3]:
model = GaussianNB()

In [4]:
import time 

trained_now = False 
training_time = None

model_loaded = rm.load_model()

if model_loaded is not None: 
    model = model_loaded
    print("Successfully loaded existing model.")
else:
    print("Training model ... ")
    start_time = time.time()

    model.fit(X_train, y_train)

    training_time = time.time() - start_time
    print(f"Training done in {training_time:.2f} seconds")

    # save model using results manager 
    rm.save_model(model)

    trained_now = True 

Path koju gledam: ../../results/naive_bayes/model.pkl
Successfully loaded existing model.


In [5]:

y_pred = model.predict(X_test)

In [6]:
from sklearn.metrics import accuracy_score, f1_score

accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average = "macro")
weighted_f1 = f1_score(y_test, y_pred, average = "weighted")

print("Accuracy:", accuracy)
print("Macro F1:", macro_f1)
print("Weighted F1:", weighted_f1)

Accuracy: 0.3133594341095512
Macro F1: 0.16686531361483908
Weighted F1: 0.3114372104579698


In [7]:
import json 

if training_time is None:
    training_time = -0.1

results = {
    "model": "naive_bayes",
    "accuracy": float(accuracy),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "training_time_seconds": float(training_time)
}

# save results in .json file only if model is trained now: 
if trained_now:
    rm.save_metrics(results)


In [8]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(f"Shape: {cm.shape}\n")
print(cm)

Shape: (256, 256)

[[  0   0   0 ...   2   0   0]
 [  0  14   6 ...   2   0   0]
 [  0  17   0 ...   0   0   0]
 ...
 [  0   0   0 ... 130   0   0]
 [  0   0   0 ...   8   6   1]
 [  0   0   0 ...   0   0   7]]


In [9]:
errors = []

n = cm.shape[0]

for true_class in range(n):
    for pred_class in range(n):
        if true_class != pred_class:            # avoid diag elements 
            count = cm[true_class, pred_class]
            errors.append((true_class, pred_class, count))

errors = sorted(errors, key = lambda x: x[2], reverse = True)

top_errors = errors[:10]

top_errors_readable = []
for true_class, pred_class, count in top_errors:
    label = f"{chr(true_class)} → {chr(pred_class)}"
    top_errors_readable.append((label, count))

errors_df = pd.DataFrame(
    top_errors_readable,
    columns = ["confusion", "count"]
)

rm.save_dataframe(errors_df, "top_confusions")

errors_df

Dataframe saved to ../../results/naive_bayes/top_confusions.csv


,confusion,count
0,1 → /,956
1,0 → O,907
2,0 → Ú,451
3,1 → 9,419
4,1 → I,387
5,1 → 6,313
6,6 → ó,305
7,4 → A,300
8,8 → î,297
9,4 → Ý,257


### Per-class accuracy 

In [10]:
# number of correct classifications per class (diag elements)
correct = np.diag(cm)

# total number of samples per class
total = cm.sum(axis = 1)

# per-class accuracy
per_class_accuracy = correct / total

# ASCII oznake klasa
labels = [f"{i}:{repr(chr(i))}" for i in range(cm.shape[0])]

# dataframe sa rezultatima
per_class_df = pd.DataFrame({
    "character": labels,
    "accuracy": per_class_accuracy,
    "correct": correct,
    "total": total,
    "percentage_%": per_class_accuracy *100
})

# sortiranje po tacnosti
per_class_df = per_class_df.sort_values("accuracy")
rm.save_dataframe(per_class_df, "class_accuracy")

worst_classified = per_class_df.head(10)
rm.save_dataframe(worst_classified, "worst_classified_classes")
print("Worst classified characters:")
print(worst_classified)

best_classified = per_class_df.tail(10)
rm.save_dataframe(best_classified, "best_classified_classes")
print("\nBest classified characters:")
print(best_classified)

Dataframe saved to ../../results/naive_bayes/class_accuracy.csv
Dataframe saved to ../../results/naive_bayes/worst_classified_classes.csv
Worst classified characters:
      character  accuracy  correct  total  percentage_%
0      0:'\x00'       0.0        0    293           0.0
108     108:'l'       0.0        0    445           0.0
120     120:'x'       0.0        0    571           0.0
121     121:'y'       0.0        0    450           0.0
124     124:'|'       0.0        0    451           0.0
129  129:'\x81'       0.0        0    212           0.0
132  132:'\x84'       0.0        0    276           0.0
134  134:'\x86'       0.0        0    220           0.0
135  135:'\x87'       0.0        0    220           0.0
136  136:'\x88'       0.0        0    234           0.0
Dataframe saved to ../../results/naive_bayes/best_classified_classes.csv

Best classified characters:
   character  accuracy  correct  total  percentage_%
76    76:'L'  0.621532      560    901     62.153163
80    80:

In [11]:
import warnings
warnings.filterwarnings("ignore")

font_results = []

for font in np.unique(font_test):
    mask = font_test == font

    acc = accuracy_score(
        y_test[mask],
        y_pred[mask]
    )

    font_results.append({
        "font": font,
        "samples": mask.sum(),
        "accuracy": acc
    })


font_df = pd.DataFrame(font_results)
font_df = font_df.sort_values("accuracy", ascending = False)
font_df["accuracy_%"] = (font_df["accuracy"] * 100).round(2)
rm.save_dataframe(font_df, "font_accuracy")

font_df

Dataframe saved to ../../results/naive_bayes/font_accuracy.csv


,font,samples,accuracy,accuracy_%
43,E13B,4718,0.851632,85.16
102,OCRB,18706,0.825404,82.54
100,NUMERICS,2780,0.796763,79.68
101,OCRA,13077,0.778084,77.81
146,VIN,218,0.733945,73.39
...,...,...,...,...
152,YI BAITI,1206,0.046434,4.64
20,CALIBRI,3835,0.044850,4.49
137,TAHOMA,2675,0.032523,3.25
123,SCRIPT,438,0.029680,2.97


In [12]:
italic_results = []

for value in np.unique(italic_test):
    mask = italic_test == value 

    acc = accuracy_score(
        y_test[mask],
        y_pred[mask]
    )

    italic_results.append({
        "italic": value, 
        "samples": mask.sum(),
        "accuracy": acc
    })

italic_df = pd.DataFrame(italic_results)
italic_df["accuracy_%"] = (italic_df["accuracy"] * 100).round(2)
rm.save_dataframe(italic_df, "italic_accuracy")

italic_df

Dataframe saved to ../../results/naive_bayes/italic_accuracy.csv


,italic,samples,accuracy,accuracy_%
0,0,116206,0.403921,40.39
1,1,50328,0.104256,10.43


57.13% karaktera koji nisu italic je klasifikovano tacno  
26.15% karaktera koji jesu italic je klasifikovano tacno

In [13]:
bins = np.linspace(0, 1, 6)   # 0.0, 0.2, 0.4, 0.6, 0.8, 1.0
labels = [f"{bins[i]:.1f}-{bins[i+1]:.1f}" for i in range(len(bins)-1)]

strength_results = []

for i in range(len(bins)-1):

    mask = (strength_test >= bins[i]) & (strength_test < bins[i+1])

    samples = mask.sum()
    if samples == 0:
        acc = np.nan
    else: 
        acc = accuracy_score(
            y_test[mask],
            y_pred[mask]
        )


    strength_results.append({
        "strength_range": labels[i],
        "samples": samples,
        "accuracy": acc
    })


strength_df = pd.DataFrame(strength_results)
strength_df["accuracy_%"] = (strength_df["accuracy"] * 100).round(2)

rm.save_dataframe(strength_df, "strength_accuracy")

strength_df

Dataframe saved to ../../results/naive_bayes/strength_accuracy.csv


,strength_range,samples,accuracy,accuracy_%
0,0.0-0.2,0,NaN,NaN
1,0.2-0.4,0,NaN,NaN
2,0.4-0.6,114924,0.400247,40.02
3,0.6-0.8,51610,0.119880,11.99
4,0.8-1.0,0,NaN,NaN


Deblji karakteri imaju više spojenih piksela, pa se oblici slova međusobno približavaju.  
Na primer: 8 vs B, 5 vs S, O vs 0 -> jos slicniji kada je font deblji

In [14]:
probs = model.predict_proba(X_test)
confidence = probs.max(axis = 1)

correct_conf = confidence[y_pred == y_test]
wrong_conf = confidence[y_pred != y_test]

print(correct_conf.mean())
print(wrong_conf.mean())

0.9882818140364489
0.940125234755608


### Naive Bayes - PCA dataset 

In [3]:
from src.load_processed import load_processed_data
dataset = load_processed_data("../../data/processed/pca_preprocessed.npz")

X_train_pca = dataset["X_train"]
X_test_pca = dataset["X_test"]

y_train_pca = dataset["y_train"]
y_test_pca = dataset["y_test"]

print("Train shape:", X_train_pca.shape)
print("Test shape:", X_test_pca.shape)
print("Number of classes:", len(np.unique(y_train_pca)))
print("Data type:", X_train_pca.dtype)

Train shape: (666136, 119)
Test shape: (166534, 119)
Number of classes: 256
Data type: float32


In [4]:
import time

model_pca = GaussianNB()

start = time.time()

model_pca.fit(X_train_pca, y_train_pca)

training_time_pca = time.time() - start
print("Pca done training in: ", training_time_pca)

/home/nemanja/.local/lib/python3.10/site-packages/sklearn/utils/multiclass.py:79: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  ys_types = set(type_of_target(x) for x in ys)


Pca done training in:  1.0837116241455078


In [5]:
y_pred_pca = model_pca.predict(X_test_pca)

In [7]:
from sklearn.metrics import accuracy_score, f1_score
accuracy_pca = accuracy_score(y_test_pca, y_pred_pca)
macro_f1_pca = f1_score(y_test_pca, y_pred_pca, average="macro")
weighted_f1_pca = f1_score(y_test_pca, y_pred_pca, average="weighted")

print("PCA Accuracy:", accuracy_pca)
print("PCA Macro F1:", macro_f1_pca)
print("PCA Weighted F1:", weighted_f1_pca)

PCA Accuracy: 0.395378721462284
PCA Macro F1: 0.2723915024148278
PCA Weighted F1: 0.3997553571505875


In [ ]:
comparison_df = pd.DataFrame({
    "Model": ["Naive Bayes", "Naive Bayes + PCA"],
    "accuracy": [accuracy, accuracy_pca],
    "Macro F1": [macro_f1, macro_f1_pca],
    "Weighted_F1": [weighted_f1, weighted_f1_pca]
})

rm.save_dataframe(comparison_df, "pca_comparison")

comparison_df

In [12]:
probs = model_pca.predict_proba(X_test_pca)
confidence = probs.max(axis = 1)

correct_conf = confidence[y_pred_pca == y_test_pca]
wrong_conf = confidence[y_pred_pca != y_test_pca]

print(correct_conf.mean())
print(wrong_conf.mean())

0.9023723257212221
0.6325638814730126
